# Chapter 8: Quadrilateral Sets and Liftings

**Source span.** Perspectives on Projective Geometry, Chapter 8, sections 8.1-8.6; printed pages 129-144; PDF pages 151-166.

**Chapter goal.** Build a computational model for quadrilateral sets on a projective line, then use it to recognize liftings, symmetry orbits, von Staudt arithmetic, slope conditions, and involutions.

The chapter begins with a simple but powerful line fact: when several point pairs lie on one line, bracket products can be reordered because every triangle area splits into a base factor and an altitude factor. That product identity becomes the algebra behind quadrilateral sets, where six points on one projective line behave as if they came from the six intersections of four lines, or dually from the six joins among four points.

This notebook does not reproduce the textbook figures or prose. It rebuilds the chapter as inspectable data. Each visual artifact has a nearby invariant: a rank drop, a bracket product, a symmetry orbit, an exact von Staudt identity, a slope equation, or an involution check. The aim is that a reader can run the notebook, change coordinates, and still see what must remain true.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives on Projective Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-08-quadrilateral-sets-and-liftings"
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
TABLES = ARTIFACT_ROOT / "tables"
CHECKS = ARTIFACT_ROOT / "checks"
for folder in (FIGURES, HTML, TABLES, CHECKS):
    folder.mkdir(parents=True, exist_ok=True)

ARTIFACT_ROOT.relative_to(BOOK_ROOT)


## Visualization Planner Pass

The planner pass routes each source idea to a representation that can be inspected and checked.

| Source idea | Computational representation | Library route | Inspection target | Invariant or check |
| --- | --- | --- | --- | --- |
| Points on one line | finite coordinates on a base line plus off-line altitudes | NumPy and Matplotlib | compare base-length factors with altitude factors | two bracket products agree after a permutation |
| Quadrilateral set | six RP^1 points grouped into three opposite pairs | SymPy, NumPy, Matplotlib | read the two bracket monomials on the same line | `[ae][bf][cd] = [af][bd][ce]` |
| Lifting | homogeneous line points lifted to `(x, h)` with four collinearity equations | NumPy linear algebra | see the four lifted triples as straight lines | lifting matrix has rank 3 and a non-affine null vector |
| Symmetry | cube of choices from `(a,d)`, `(b,e)`, `(c,f)` | NetworkX and Matplotlib | even and odd parity vertices are complementary quadrilaterals | pair swaps preserve the quadset condition |
| von Staudt links | addition and multiplication as two special quadsets | SymPy exact algebra | see arithmetic as pair data on a projective line | symbolic residuals simplify to zero |
| Slope conditions | slopes of the six lines through four finite points | NumPy, Matplotlib, Plotly | identify opposite line pairs and the moving slope line | slope quadset residual is zero under perturbation |
| Involutions | three point pairs generated by a 2 by 2 projective matrix | NumPy, SymPy, Matplotlib | compare reflection-like and rotation-like maps | `T^2` is scalar identity and generated pairs form a quadset |

The proof scaffold is a directed dependency graph: line product identity -> quadrilateral bracket identity -> rank-drop lifting -> symmetry and arithmetic uses -> slope and involution consequences. The static figures are durable PNG files; the perturbation lab is a standalone Plotly HTML file; exact and numeric validation data are saved as JSON and CSV under the chapter artifact subtree.


In [ ]:
import json
from fractions import Fraction

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from matplotlib.patches import FancyArrowPatch
from plotly.subplots import make_subplots

from utils.artifacts import assert_artifacts, book_relative, display_artifact, save_json, save_table

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLORS = {
    "left": "#2a6fbb",
    "right": "#c94f4f",
    "line": "#2f2f2f",
    "accent": "#268a5e",
    "gold": "#c08a1a",
    "muted": "#6d7280",
}

storyboard = {
    "chapter goal": "Model quadrilateral sets on RP^1 and connect them to liftings, von Staudt arithmetic, slopes, and involutions.",
    "source span read": "Chapter 8 sections 8.1-8.6; printed pages 129-144; PDF pages 151-166.",
    "concept inventory": [
        "points on a line and bracket product reorderings",
        "six-point quadrilateral set condition",
        "lifting as a rank drop in four collinearity equations",
        "three opposite point pairs and the even/odd symmetry orbit",
        "von Staudt addition and multiplication as special quadrilateral sets",
        "slope condition from four finite points",
        "projective involutions and harmonic fixed-point scaffold",
    ],
    "library routing table": [
        {"concept": "line product identity", "representation": "labeled 2D diagram plus product table", "library": "NumPy, Matplotlib", "why": "the geometry is planar incidence and area factoring"},
        {"concept": "quadrilateral set and lifting", "representation": "RP^1 bracket plot and nullspace lifting", "library": "SymPy, NumPy, Matplotlib", "why": "exact brackets and rank checks expose the algebra"},
        {"concept": "symmetry orbit", "representation": "cube/dependency graph", "library": "NetworkX, Matplotlib", "why": "pair swaps are graph moves, not decoration"},
        {"concept": "slope perturbation", "representation": "interactive line/slope lab", "library": "Plotly", "why": "the learner can hover and scrub a moving vertex"},
        {"concept": "von Staudt and involutions", "representation": "symbolic checks plus line diagrams", "library": "SymPy, NumPy, Matplotlib", "why": "the central claims are algebraic identities on RP^1"},
    ],
    "visual sequence": [
        "visual-route-map.png",
        "points-on-line-product-identity.png",
        "quadrilateral-bracket-balance.png",
        "lifting-rank-nullspace.png",
        "symmetry-pair-swap-orbits.png",
        "von-staudt-quadset-links.png",
        "slope-condition-opposite-pairs.png",
        "involution-pairs-fixed-points.png",
        "slope-quadset-motion-lab.html",
    ],
    "computational checks": [
        "line-product residual below tolerance",
        "quadrilateral bracket residual below tolerance and exact symbolic residuals equal zero",
        "lifting matrix rank is 3 and collinearity residuals vanish",
        "slope perturbation residuals stay below tolerance",
        "involution matrix square is scalar identity and generated pairs form a quadrilateral set",
        "all displayed artifacts exist and have nonzero size",
    ],
    "proof-visualization strategy": "Use a dependency graph plus invariant tables rather than copied proof text.",
    "risks": ["degenerate coincident points", "vertical slope cases", "stale artifact paths"],
    "acceptance criteria": ["direct nbclient execution", "chapter-scoped validation", "scoped visual/notebook audits", "git diff --check"],
}
save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")


def figure_path(name):
    return FIGURES / name


def html_path(name):
    return HTML / name


def save_figure(fig, name):
    path = figure_path(name)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def rp1(value):
    if value is None or value == "inf":
        return sp.Matrix([1, 0])
    return sp.Matrix([sp.sympify(value), 1])


def bracket2(u, v):
    u = rp1(u) if not isinstance(u, sp.MatrixBase) else u
    v = rp1(v) if not isinstance(v, sp.MatrixBase) else v
    return sp.simplify(u[0] * v[1] - u[1] * v[0])


def quadset_terms(a, b, c, d, e, f):
    left = bracket2(a, e) * bracket2(b, f) * bracket2(c, d)
    right = bracket2(a, f) * bracket2(b, d) * bracket2(c, e)
    return sp.simplify(left), sp.simplify(right), sp.simplify(left - right)


def quadset_residual(values):
    a, b, c, d, e, f = [sp.Rational(values[k]).limit_denominator() for k in ["a", "b", "c", "d", "e", "f"]]
    left, right, residual = quadset_terms(a, b, c, d, e, f)
    return float(sp.N(left)), float(sp.N(right)), float(sp.N(residual))


def lifting_matrix(values):
    labels = ["a", "b", "c", "d", "e", "f"]
    triples = [("a", "b", "c"), ("a", "e", "f"), ("b", "d", "f"), ("c", "d", "e")]
    matrix = np.zeros((4, 6), dtype=float)
    for row, (i, j, k) in enumerate(triples):
        xi, xj, xk = values[i], values[j], values[k]
        matrix[row, labels.index(k)] = xi - xj
        matrix[row, labels.index(j)] = -(xi - xk)
        matrix[row, labels.index(i)] = xj - xk
    return matrix, labels, triples


def finite_slope(p, q):
    dx = q[0] - p[0]
    dy = q[1] - p[1]
    if abs(dx) < 1e-12:
        raise ValueError("vertical slope avoided in this chapter lab")
    return dy / dx


def line_axis(ax, xmin, xmax, y=0, label="projective line"):
    ax.plot([xmin, xmax], [y, y], color=COLORS["line"], lw=1.6)
    ax.text(xmin, y - 0.13, label, ha="left", va="top", color=COLORS["muted"])
    ax.set_ylim(y - 0.55, y + 1.55)
    ax.set_yticks([])
    ax.set_xlabel("affine coordinate on RP^1 chart")


def annotate_point(ax, x, label, y=0, color="#111111", dy=0.11):
    ax.scatter([x], [y], s=52, color=color, zorder=5, edgecolor="white", linewidth=0.8)
    ax.text(x, y + dy, label, ha="center", va="bottom", color=color, fontweight="bold")


## Points on a Line: Product Identity

A point on a projective line can be represented in an affine chart by a single coordinate. If two points `a_i` and `b_i` lie on the same base line and a third point `x_i` is off that line, the determinant bracket `[a_i b_i x_i]` factors into an oriented base length times an altitude. A product of such brackets is therefore insensitive to reordering the off-line points: the same base factors and the same altitude factors appear, only paired in a different order.

The figure below makes that factoring visible. The bottom intervals are the `[a_i b_i]` factors; the vertical ticks encode the altitude factors. The check records that the product before and after a permutation agrees.


In [ ]:
base_pairs = [(-2.8, -1.6), (-0.8, 0.3), (1.1, 2.5), (3.1, 4.0)]
heights = np.array([1.15, 0.72, 1.48, 0.95])
perm = [2, 0, 3, 1]
base_lengths = np.array([b - a for a, b in base_pairs])
line_product_left = float(np.prod(base_lengths * heights))
line_product_right = float(np.prod(base_lengths * heights[perm]))
line_product_residual = abs(line_product_left - line_product_right)

fig, ax = plt.subplots(figsize=(9.2, 3.6))
line_axis(ax, -3.1, 4.3, label="common line carrying all a_i and b_i")
for i, ((a0, b0), h) in enumerate(zip(base_pairs, heights), start=1):
    color = [COLORS["left"], COLORS["accent"], COLORS["gold"], COLORS["right"]][i - 1]
    ax.plot([a0, b0], [0, 0], lw=5, color=color, solid_capstyle="round")
    ax.plot([(a0 + b0) / 2, (a0 + b0) / 2], [0, h], lw=1.5, color=color, ls="--")
    ax.scatter([(a0 + b0) / 2], [h], s=58, color=color, edgecolor="white", zorder=6)
    ax.text(a0, -0.17, f"a{i}", ha="center")
    ax.text(b0, -0.17, f"b{i}", ha="center")
    ax.text((a0 + b0) / 2, h + 0.08, f"x{i}", ha="center", color=color, fontweight="bold")
ax.set_title("A bracket product on one line factors into base lengths and altitudes")
ax.text(-3.05, 1.32, f"product residual after permutation = {line_product_residual:.2e}", color=COLORS["muted"])
points_identity_path = save_figure(fig, "points-on-line-product-identity.png")

line_identity_row = {
    "concept": "points on a line product identity",
    "left_product": line_product_left,
    "right_product": line_product_right,
    "residual": line_product_residual,
    "artifact": book_relative(points_identity_path),
}
line_identity_row


## Quadrilateral Sets and Liftings

A quadrilateral set is six points on `RP^1` grouped into three opposite pairs `(a,d)`, `(b,e)`, and `(c,f)` such that

`[a e][b f][c d] = [a f][b d][c e]`.

Here `[u v]` is the 2 by 2 determinant of the two homogeneous line coordinates. The same equation appears in two useful ways. First, it characterizes the six projected intersections of four lines. Second, it says that the four requested triples `(a,b,c)`, `(a,e,f)`, `(b,d,f)`, and `(c,d,e)` can be lifted from a line into a nontrivial planar drawing where each triple becomes collinear.

The data below comes from slopes of six lines through four finite points, so it is guaranteed to satisfy the quadrilateral relation. We reuse it here because it keeps the line picture, the lifting picture, and the slope picture tied to the same checked numbers.


In [ ]:
vertices = {
    "P0": np.array([0.0, 0.0]),
    "P1": np.array([12.0, 7.0]),
    "P2": np.array([-6.0, 15.0]),
    "P3": np.array([21.0, -4.0]),
}
slopes = {
    "a": Fraction(7, 12),
    "d": Fraction(-19, 27),
    "b": Fraction(-5, 2),
    "e": Fraction(-11, 9),
    "c": Fraction(-4, 21),
    "f": Fraction(-4, 9),
}
quad_values = {k: float(v) for k, v in slopes.items()}
quad_left, quad_right, quad_residual = quadset_residual(slopes)

fig, ax = plt.subplots(figsize=(9.5, 3.3))
xs = [quad_values[k] for k in ["b", "e", "d", "f", "c", "a"]]
line_axis(ax, min(xs) - 0.35, max(xs) + 0.35, label="six points on one projective line")
for label in ["a", "b", "c", "d", "e", "f"]:
    color = COLORS["left"] if label in {"a", "b", "c"} else COLORS["right"]
    annotate_point(ax, quad_values[label], label, color=color)
for first, second, height in [("a", "e", 0.45), ("b", "f", 0.70), ("c", "d", 0.95)]:
    x0, x1 = quad_values[first], quad_values[second]
    ax.add_patch(FancyArrowPatch((x0, height), (x1, height), arrowstyle="<->", mutation_scale=10, lw=1.4, color=COLORS["left"], connectionstyle="arc3,rad=0.12"))
    ax.text((x0 + x1) / 2, height + 0.06, f"[{first}{second}]", ha="center", color=COLORS["left"])
for first, second, height in [("a", "f", 1.20), ("b", "d", 1.42), ("c", "e", 1.04)]:
    x0, x1 = quad_values[first], quad_values[second]
    ax.add_patch(FancyArrowPatch((x0, height), (x1, height), arrowstyle="<->", mutation_scale=10, lw=1.4, color=COLORS["right"], connectionstyle="arc3,rad=-0.12"))
    ax.text((x0 + x1) / 2, height + 0.06, f"[{first}{second}]", ha="center", color=COLORS["right"])
ax.set_title("Quadrilateral set bracket balance")
ax.text(min(xs) - 0.3, 1.42, f"left - right = {quad_residual:.2e}", color=COLORS["muted"])
quad_balance_path = save_figure(fig, "quadrilateral-bracket-balance.png")

M, lift_labels, lift_triples = lifting_matrix(quad_values)
rank = int(np.linalg.matrix_rank(M, tol=1e-10))
v1 = np.array([0, 111/14, 195/98, 1, 0, 0], dtype=float)
v2 = np.array([-37/28, 0, -97/98, 0, 1, 0], dtype=float)
v3 = np.array([65/28, -97/14, 0, 0, 0, 1], dtype=float)
heights_lift = v1 - 0.5 * v2 + 0.8 * v3
heights_lift = heights_lift / np.max(np.abs(heights_lift)) * 1.5
lift_residuals = M @ heights_lift

fig, ax = plt.subplots(figsize=(9.3, 5.0))
order = ["b", "e", "d", "f", "c", "a"]
for label in order:
    ax.plot([quad_values[label], quad_values[label]], [0, heights_lift[lift_labels.index(label)]], color="#c7cbd1", ls=":", lw=1)
    annotate_point(ax, quad_values[label], label, y=0, color="#333333", dy=-0.22)
    ax.scatter([quad_values[label]], [heights_lift[lift_labels.index(label)]], s=70, color=COLORS["accent"], edgecolor="white", zorder=5)
    ax.text(quad_values[label], heights_lift[lift_labels.index(label)] + 0.08, label.upper(), ha="center", color=COLORS["accent"], fontweight="bold")
colors = [COLORS["left"], COLORS["right"], COLORS["gold"], COLORS["accent"]]
for color, triple in zip(colors, lift_triples):
    pts = sorted([(quad_values[k], heights_lift[lift_labels.index(k)]) for k in triple])
    ax.plot([p[0] for p in pts], [p[1] for p in pts], color=color, lw=2.0, label="".join(triple))
line_axis(ax, min(xs) - 0.35, max(xs) + 0.35, label="original line before lifting")
ax.set_ylabel("lifting height h")
ax.set_title("A rank-3 lifting matrix gives a nontrivial collinear lifting")
ax.legend(title="lifted triples", loc="upper left", frameon=False)
ax.text(min(xs) - 0.3, 1.48, f"rank(M) = {rank}; max collinearity residual = {np.max(np.abs(lift_residuals)):.2e}", color=COLORS["muted"])
lifting_path = save_figure(fig, "lifting-rank-nullspace.png")

quad_rows = [
    {"concept": "quadrilateral bracket balance", "left": quad_left, "right": quad_right, "residual": quad_residual, "artifact": book_relative(quad_balance_path)},
    {"concept": "lifting matrix rank", "rank": rank, "max_residual": float(np.max(np.abs(lift_residuals))), "artifact": book_relative(lifting_path)},
]
quad_rows


## Symmetry Orbits

The labels are not arbitrary. The opposite pairs `(a,d)`, `(b,e)`, and `(c,f)` determine a cube of eight triples, because each triple chooses one point from each pair. The four even-parity triples are one complete quadrilateral; the four odd-parity triples are the associated complete quadrilateral. Swapping one point inside a pair moves across an edge of the cube and exchanges the two quadrilaterals.

This is the useful symmetry fact: the same bracket condition does not belong to only one drawing. It belongs to the three-pair structure.


In [ ]:
route_edges = [
    ("line product\nidentity", "quadset\nbracket balance"),
    ("quadset\nbracket balance", "rank-drop\nlifting"),
    ("quadset\nbracket balance", "pair-swap\nsymmetry"),
    ("quadset\nbracket balance", "von Staudt\narithmetic"),
    ("von Staudt\narithmetic", "slope\ncondition"),
    ("slope\ncondition", "projective\ninvolution"),
    ("projective\ninvolution", "fixed-point\nharmonic scaffold"),
]
G_route = nx.DiGraph(route_edges)
pos_route = {
    "line product\nidentity": (0, 1.1),
    "quadset\nbracket balance": (1.5, 1.1),
    "rank-drop\nlifting": (3.0, 1.75),
    "pair-swap\nsymmetry": (3.0, 0.45),
    "von Staudt\narithmetic": (4.6, 1.1),
    "slope\ncondition": (6.1, 1.1),
    "projective\ninvolution": (7.6, 1.1),
    "fixed-point\nharmonic scaffold": (9.2, 1.1),
}
fig, ax = plt.subplots(figsize=(11, 3.2))
nx.draw_networkx_edges(G_route, pos_route, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=15, width=1.6, edge_color="#657188")
nx.draw_networkx_nodes(G_route, pos_route, ax=ax, node_size=2900, node_color="#f3f6fa", edgecolors="#2d3b52", linewidths=1.2)
nx.draw_networkx_labels(G_route, pos_route, ax=ax, font_size=8)
ax.set_title("Chapter 8 route: one bracket identity drives several geometric readings")
ax.set_axis_off()
route_path = save_figure(fig, "visual-route-map.png")

pairs = [("a", "d"), ("b", "e"), ("c", "f")]
G = nx.Graph()
labels = {}
for bits in [(i, j, k) for i in [0, 1] for j in [0, 1] for k in [0, 1]]:
    triple = "".join(pairs[idx][bit] for idx, bit in enumerate(bits))
    parity = sum(bits) % 2
    G.add_node(bits, triple=triple, parity=parity)
    labels[bits] = triple
for node in G.nodes:
    for i in range(3):
        other = list(node)
        other[i] = 1 - other[i]
        other = tuple(other)
        if node < other:
            G.add_edge(node, other)
pos = {node: (node[0] + 0.18 * node[2], node[1] + 0.18 * node[2]) for node in G.nodes}
node_colors = ["#2a6fbb" if G.nodes[n]["parity"] == 0 else "#268a5e" for n in G.nodes]
fig, ax = plt.subplots(figsize=(6.8, 5.0))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#9aa3ad", width=1.2)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=1050, edgecolors="white", linewidths=1.6)
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_color="white", font_weight="bold")
ax.text(-0.12, 1.35, "even parity: original complete quadrilateral", color="#2a6fbb")
ax.text(-0.12, 1.20, "odd parity: associated complete quadrilateral", color="#268a5e")
ax.set_title("Pair-swap orbit from (a,d), (b,e), (c,f)")
ax.set_axis_off()
symmetry_path = save_figure(fig, "symmetry-pair-swap-orbits.png")

symmetry_rows = [
    {"triple": G.nodes[node]["triple"], "parity": "even-original" if G.nodes[node]["parity"] == 0 else "odd-associated", "chosen_from_pairs": str(node)}
    for node in sorted(G.nodes)
]
symmetry_rows[:3]


## von Staudt Links

Von Staudt constructions make arithmetic on a projective line from incidence alone. In this chapter they reappear as special quadrilateral sets. With a basis labelled `0`, `1`, and `infinity`, addition is encoded by the pair data `(0, x + y)`, `(x, y)`, `(infinity, infinity)`, and multiplication is encoded by `(0, infinity)`, `(x, y)`, `(1, x*y)`.

The important point is not that a diagram computes numbers by magic. The point is that the quadrilateral set identity is already the arithmetic identity. The symbolic checks below keep the point at infinity as a homogeneous vector, so no limiting argument is hidden.


In [ ]:
x, y = sp.symbols("x y")
add_left, add_right, add_residual = quadset_terms(0, x, "inf", x + y, y, "inf")
mul_left, mul_right, mul_residual = quadset_terms(0, x, 1, "inf", y, x * y)
assert sp.simplify(add_residual) == 0
assert sp.simplify(mul_residual) == 0

x0, y0 = sp.Rational(3, 5), sp.Rational(7, 4)
add_values = {"0": 0.0, "x": float(x0), "y": float(y0), "x+y": float(x0 + y0)}
mul_values = {"0": 0.0, "1": 1.0, "x": float(x0), "y": float(y0), "x*y": float(x0 * y0)}

fig, axes = plt.subplots(2, 1, figsize=(9.2, 4.8), sharex=False)
for ax, title, values, pairs_here in [
    (axes[0], "addition quadset: (0, x+y), (x, y), (infinity, infinity)", add_values, [("0", "x+y"), ("x", "y")]),
    (axes[1], "multiplication quadset: (0, infinity), (x, y), (1, x*y)", mul_values, [("x", "y"), ("1", "x*y")]),
]:
    finite_values = list(values.values())
    line_axis(ax, min(finite_values) - 0.25, max(finite_values) + 0.45, label="finite chart; infinity is the direction point")
    for label, value in values.items():
        annotate_point(ax, value, label, color=COLORS["left"] if label in {"0", "1", "x"} else COLORS["right"])
    for idx, (u, v) in enumerate(pairs_here):
        x0p, x1p = values[u], values[v]
        h = 0.55 + 0.35 * idx
        ax.add_patch(FancyArrowPatch((x0p, h), (x1p, h), arrowstyle="<->", mutation_scale=10, color=COLORS["accent"], lw=1.4))
        ax.text((x0p + x1p) / 2, h + 0.05, f"({u}, {v})", ha="center", color=COLORS["accent"])
    ax.text(max(finite_values) + 0.18, 0.58, "infinity", color=COLORS["muted"], ha="left")
    ax.set_title(title, fontsize=10)
fig.suptitle("von Staudt arithmetic appears as two quadrilateral-set identities", y=1.02)
von_staudt_path = save_figure(fig, "von-staudt-quadset-links.png")

von_staudt_rows = [
    {"concept": "von Staudt addition", "symbolic_residual": str(sp.simplify(add_residual)), "artifact": book_relative(von_staudt_path)},
    {"concept": "von Staudt multiplication", "symbolic_residual": str(sp.simplify(mul_residual)), "artifact": book_relative(von_staudt_path)},
]
von_staudt_rows


## Slope Conditions

Intersecting the six joins among four finite points with the line at infinity records their slopes. Opposite sides of the complete quadrilateral become the opposite pairs `(a,d)`, `(b,e)`, and `(c,f)`. In a finite slope chart, the quadrilateral identity becomes `(a-e)(b-f)(c-d) = (a-f)(b-d)(c-e)`.

This is a projective statement read through an affine coordinate choice. The left panel below shows the four points and their six connecting lines. The right panel shows only their slopes, which still remember the quadrilateral set.


In [ ]:
segments = {
    "a": ("P0", "P1"),
    "d": ("P2", "P3"),
    "b": ("P0", "P2"),
    "e": ("P1", "P3"),
    "c": ("P0", "P3"),
    "f": ("P1", "P2"),
}
segment_colors = {"a": COLORS["left"], "d": COLORS["left"], "b": COLORS["gold"], "e": COLORS["gold"], "c": COLORS["accent"], "f": COLORS["accent"]}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
for label, (u, v) in segments.items():
    p, q = vertices[u], vertices[v]
    ax.plot([p[0], q[0]], [p[1], q[1]], lw=2.0, color=segment_colors[label], alpha=0.85)
    mid = (p + q) / 2
    ax.text(mid[0], mid[1], label, color=segment_colors[label], fontweight="bold", ha="center", va="center", bbox={"fc": "white", "ec": "none", "alpha": 0.72, "pad": 1.0})
for label, point in vertices.items():
    ax.scatter([point[0]], [point[1]], s=70, color="#222222", edgecolor="white", zorder=5)
    ax.text(point[0] + 0.35, point[1] + 0.35, label, fontweight="bold")
ax.set_aspect("equal", adjustable="box")
ax.set_title("Four finite points determine six line slopes")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.18)

ax = axes[1]
slope_order = sorted(quad_values, key=quad_values.get)
line_axis(ax, min(xs) - 0.35, max(xs) + 0.35, label="slope chart on the line at infinity")
for label in slope_order:
    annotate_point(ax, quad_values[label], label, color=segment_colors[label])
for (u, v, h, color) in [("a", "d", 0.48, COLORS["left"]), ("b", "e", 0.78, COLORS["gold"]), ("c", "f", 1.08, COLORS["accent"])]:
    ax.add_patch(FancyArrowPatch((quad_values[u], h), (quad_values[v], h), arrowstyle="<->", mutation_scale=10, lw=1.4, color=color))
    ax.text((quad_values[u] + quad_values[v]) / 2, h + 0.06, f"opposite pair ({u},{v})", ha="center", color=color)
ax.set_title(f"slope residual = {quad_residual:.2e}")
slope_path = save_figure(fig, "slope-condition-opposite-pairs.png")

x_positions = np.linspace(16.0, 26.0, 16)
base_vertices = {k: v.copy() for k, v in vertices.items()}

def slopes_from_vertices(vdict):
    return {
        "a": finite_slope(vdict["P0"], vdict["P1"]),
        "d": finite_slope(vdict["P2"], vdict["P3"]),
        "b": finite_slope(vdict["P0"], vdict["P2"]),
        "e": finite_slope(vdict["P1"], vdict["P3"]),
        "c": finite_slope(vdict["P0"], vdict["P3"]),
        "f": finite_slope(vdict["P1"], vdict["P2"]),
    }

def frame_data(px):
    vdict = {k: v.copy() for k, v in base_vertices.items()}
    vdict["P3"] = np.array([px, -4.0])
    sdict = slopes_from_vertices(vdict)
    _, _, residual = quadset_residual({k: Fraction(sdict[k]).limit_denominator(10000) for k in sdict})
    seg_x, seg_y = [], []
    for label, (u, v) in segments.items():
        p, q = vdict[u], vdict[v]
        seg_x += [p[0], q[0], None]
        seg_y += [p[1], q[1], None]
    point_x = [vdict[k][0] for k in ["P0", "P1", "P2", "P3"]]
    point_y = [vdict[k][1] for k in ["P0", "P1", "P2", "P3"]]
    slope_x = [sdict[k] for k in ["a", "b", "c", "d", "e", "f"]]
    slope_y = [0] * 6
    title = f"Move P3 horizontally: P3.x={px:.1f}, quadset residual={abs(residual):.2e}"
    return seg_x, seg_y, point_x, point_y, slope_x, slope_y, title

seg_x, seg_y, point_x, point_y, slope_x, slope_y, initial_title = frame_data(21.0)
fig_plotly = make_subplots(rows=1, cols=2, subplot_titles=("finite four-point drawing", "six slopes on RP^1"))
fig_plotly.add_trace(go.Scatter(x=seg_x, y=seg_y, mode="lines", line=dict(width=3, color="#53657d"), hoverinfo="skip", name="six joins"), row=1, col=1)
fig_plotly.add_trace(go.Scatter(x=point_x, y=point_y, mode="markers+text", text=["P0", "P1", "P2", "P3"], textposition="top center", marker=dict(size=11, color="#222222"), name="points"), row=1, col=1)
fig_plotly.add_trace(go.Scatter(x=slope_x, y=slope_y, mode="markers+text", text=["a", "b", "c", "d", "e", "f"], textposition="top center", marker=dict(size=12, color=[segment_colors[k] for k in ["a", "b", "c", "d", "e", "f"]]), name="slopes"), row=1, col=2)
frames = []
for px in x_positions:
    sx, sy, pxs, pys, slx, sly, title = frame_data(float(px))
    frames.append(go.Frame(data=[go.Scatter(x=sx, y=sy), go.Scatter(x=pxs, y=pys), go.Scatter(x=slx, y=sly)], name=f"{px:.1f}", layout=go.Layout(title_text=title)))
fig_plotly.frames = frames
fig_plotly.update_layout(
    title=initial_title,
    showlegend=False,
    width=960,
    height=480,
    sliders=[{
        "steps": [{"args": [[frame.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": frame.name, "method": "animate"} for frame in frames],
        "currentvalue": {"prefix": "P3.x = "},
    }],
)
fig_plotly.update_xaxes(title_text="x", row=1, col=1)
fig_plotly.update_yaxes(title_text="y", scaleanchor="x", scaleratio=1, row=1, col=1)
fig_plotly.update_xaxes(title_text="slope", row=1, col=2)
fig_plotly.update_yaxes(visible=False, row=1, col=2)
plotly_path = html_path("slope-quadset-motion-lab.html")
fig_plotly.write_html(plotly_path, include_plotlyjs=True, full_html=True)

slope_rows = []
for px in x_positions:
    vdict = {k: v.copy() for k, v in base_vertices.items()}
    vdict["P3"] = np.array([float(px), -4.0])
    sdict = slopes_from_vertices(vdict)
    _, _, residual = quadset_residual({k: Fraction(sdict[k]).limit_denominator(10000) for k in sdict})
    slope_rows.append({"p3_x": float(px), "quadset_residual": float(abs(residual))})
slope_lab_table = save_table(slope_rows, ARTIFACT_ROOT, "tables", "slope-perturbation-lab.csv")

{"slope_artifact": book_relative(slope_path), "interactive_lab": book_relative(plotly_path), "max_motion_residual": max(row["quadset_residual"] for row in slope_rows)}


## Involutions and Quadrilateral Sets

A projective involution on `RP^1` is represented by a 2 by 2 matrix `T` whose square is a scalar multiple of the identity. If `T` is not the identity, then any three pairs `(a,T(a))`, `(b,T(b))`, and `(c,T(c))` form a quadrilateral set.

There are two useful mental models. A reflection-like involution has two real fixed points; those fixed points form harmonic pairs with every point pair in the quadrilateral set. A rotation-like involution can have no real fixed points, as with the map `t -> -1/t`. The figure focuses on the reflection-like map `t -> 1/t`, where the fixed points are visible at `t = -1` and `t = 1`.


In [ ]:
def mobius_apply(T, t):
    vec = np.array([float(t), 1.0])
    image = T @ vec
    if abs(image[1]) < 1e-12:
        return np.inf
    return float(image[0] / image[1])

T_reflect = np.array([[0.0, 1.0], [1.0, 0.0]])
T_rotate = np.array([[0.0, -1.0], [1.0, 0.0]])
reflect_square = T_reflect @ T_reflect
rotate_square = T_rotate @ T_rotate
involution_values = {
    "a": Fraction(-3, 1), "d": Fraction(-1, 3),
    "b": Fraction(-2, 1), "e": Fraction(-1, 2),
    "c": Fraction(3, 1), "f": Fraction(1, 3),
}
inv_left, inv_right, inv_residual = quadset_residual(involution_values)

fig, axes = plt.subplots(2, 1, figsize=(9.4, 5.2), sharex=True)
all_inv_x = [float(v) for v in involution_values.values()] + [-1, 1]
for ax, title, T, fixed_note in [
    (axes[0], "reflection-like involution t -> 1/t", T_reflect, "real fixed points at -1 and 1"),
    (axes[1], "rotation-like involution t -> -1/t", T_rotate, "no real fixed point in this finite chart"),
]:
    line_axis(ax, min(all_inv_x) - 0.4, max(all_inv_x) + 0.4, label="RP^1 affine chart")
    if T is T_reflect:
        for label in ["a", "b", "c", "d", "e", "f"]:
            color = COLORS["left"] if label in {"a", "b", "c"} else COLORS["right"]
            annotate_point(ax, float(involution_values[label]), label, color=color)
        for u, v, h in [("a", "d", 0.52), ("b", "e", 0.82), ("c", "f", 1.12)]:
            ax.add_patch(FancyArrowPatch((float(involution_values[u]), h), (float(involution_values[v]), h), arrowstyle="<->", mutation_scale=10, lw=1.4, color=COLORS["accent"]))
            ax.text((float(involution_values[u]) + float(involution_values[v])) / 2, h + 0.06, f"{u} <-> {v}", ha="center", color=COLORS["accent"])
        for fixed in [-1, 1]:
            ax.scatter([fixed], [0], s=90, marker="s", color=COLORS["gold"], edgecolor="white", zorder=6)
            ax.text(fixed, -0.23, f"fixed {fixed}", ha="center", color=COLORS["gold"])
        ax.text(min(all_inv_x) - 0.25, 1.32, f"quadset residual = {inv_residual:.2e}", color=COLORS["muted"])
    else:
        for t in [-3, -1.2, -0.4, 0.4, 1.2, 3]:
            image = mobius_apply(T, t)
            if np.isfinite(image) and min(all_inv_x) - 0.4 < image < max(all_inv_x) + 0.4:
                ax.add_patch(FancyArrowPatch((t, 0.55), (image, 0.55), arrowstyle="->", mutation_scale=10, lw=1.0, color=COLORS["right"], alpha=0.75))
                annotate_point(ax, t, f"{t:g}", color=COLORS["muted"], dy=0.1)
        ax.text(min(all_inv_x) - 0.25, 1.02, fixed_note, color=COLORS["muted"])
    ax.set_title(title + " - " + fixed_note)
fig.suptitle("Involutions generate quadrilateral sets from three point pairs", y=1.02)
involution_path = save_figure(fig, "involution-pairs-fixed-points.png")

a_val = involution_values["a"]
d_val = involution_values["d"]
p_val, q_val = sp.Rational(-1), sp.Rational(1)
harmonic_residual = sp.simplify(bracket2(a_val, p_val) * bracket2(q_val, d_val) + bracket2(a_val, q_val) * bracket2(p_val, d_val))
assert harmonic_residual == 0

involution_rows = [
    {"concept": "reflection-like involution", "matrix_square": reflect_square.tolist(), "quadset_residual": inv_residual, "harmonic_residual": str(harmonic_residual), "artifact": book_relative(involution_path)},
    {"concept": "rotation-like involution", "matrix_square": rotate_square.tolist(), "fixed_point_note": "complex eigenvectors; no real fixed points", "artifact": book_relative(involution_path)},
]
involution_rows


## Applied Lab: Five Points Determine the Sixth

A quadrilateral set condition is also a reconstruction rule. If five of the six finite line coordinates are known and the denominator is nonzero, the sixth coordinate is forced. Solving `(a-e)(b-f)(c-d) = (a-f)(b-d)(c-e)` for `f` gives a small design lab: move `e`, recompute `f`, and watch the lifting rank. Degenerate choices are not mysterious; they are exactly the cases where the solve denominator vanishes or points collide.


In [ ]:
def solve_sixth_f(a, b, c, d, e):
    L = (a - e) * (c - d)
    R = (b - d) * (c - e)
    if abs(R - L) < 1e-12:
        return np.nan
    return (R * a - L * b) / (R - L)

lab_rows = []
for e_value in np.linspace(-1.7, -0.65, 9):
    trial = {"a": quad_values["a"], "b": quad_values["b"], "c": quad_values["c"], "d": quad_values["d"], "e": float(e_value)}
    trial["f"] = solve_sixth_f(trial["a"], trial["b"], trial["c"], trial["d"], trial["e"])
    if np.isfinite(trial["f"]):
        left, right, residual = quadset_residual({k: Fraction(trial[k]).limit_denominator(10000) for k in trial})
        M_trial, _, _ = lifting_matrix(trial)
        rank_trial = int(np.linalg.matrix_rank(M_trial, tol=1e-8))
    else:
        residual, rank_trial = np.nan, -1
    lab_rows.append({"e": trial["e"], "forced_f": trial["f"], "quadset_residual": abs(float(residual)), "lifting_rank": rank_trial})
lab_df = pd.DataFrame(lab_rows)
forced_table_path = save_table(lab_rows, ARTIFACT_ROOT, "tables", "five-points-force-sixth.csv")
lab_df


## Artifact Gallery

The generated artifacts are intentionally named by concept. They are displayed inline during execution and linked here for non-executed reading.

![Chapter route through the quadrilateral-set machinery](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/figures/visual-route-map.png)

![Product identity for points on one line](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/figures/points-on-line-product-identity.png)

![Quadrilateral bracket balance](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/figures/quadrilateral-bracket-balance.png)

![Rank-drop lifting](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/figures/lifting-rank-nullspace.png)

![Pair-swap symmetry orbits](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/figures/symmetry-pair-swap-orbits.png)

![von Staudt quadrilateral links](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/figures/von-staudt-quadset-links.png)

![Slope condition through opposite pairs](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/figures/slope-condition-opposite-pairs.png)

![Involution pairs and fixed points](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/figures/involution-pairs-fixed-points.png)

Open the interactive [slope quadset motion lab](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/html/slope-quadset-motion-lab.html), the [five-points-force-sixth table](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/tables/five-points-force-sixth.csv), and the [slope perturbation table](../../artifacts/chapter-08-quadrilateral-sets-and-liftings/tables/slope-perturbation-lab.csv) when reading outside a live kernel.


In [ ]:
display_paths = [
    route_path,
    points_identity_path,
    quad_balance_path,
    lifting_path,
    symmetry_path,
    von_staudt_path,
    slope_path,
    involution_path,
    plotly_path,
    forced_table_path,
    slope_lab_table,
]
for artifact in display_paths:
    display_artifact(artifact, width=760, height=430)


## Final Sanity Checks

The final cell checks three layers. First, it asserts the mathematical invariants: product identity, quadrilateral balance, symbolic von Staudt identities, lifting rank, slope perturbation, and involution/harmonic residuals. Second, it verifies that every displayed artifact exists and has nonzero size. Third, it writes a JSON record with relative paths only, so future audits can catch stale artifacts without depending on this workstation path.


In [ ]:
artifact_sizes = {book_relative(path): path.stat().st_size for path in display_paths}
assert_artifacts(display_paths)
assert line_product_residual < 1e-12
assert abs(quad_residual) < 1e-12
assert rank == 3
assert float(np.max(np.abs(lift_residuals))) < 1e-9
assert sp.simplify(add_residual) == 0
assert sp.simplify(mul_residual) == 0
assert max(row["quadset_residual"] for row in slope_rows) < 1e-8
assert abs(inv_residual) < 1e-12
assert harmonic_residual == 0
assert np.allclose(reflect_square, np.eye(2))
assert np.allclose(rotate_square, -np.eye(2))

validation_rows = []
validation_rows.extend([line_identity_row])
validation_rows.extend(quad_rows)
validation_rows.extend(von_staudt_rows)
validation_rows.extend(involution_rows)
validation_rows.append({"concept": "slope motion lab", "max_residual": max(row["quadset_residual"] for row in slope_rows), "artifact": book_relative(plotly_path)})
validation_rows.append({"concept": "five points force sixth", "max_residual": float(lab_df["quadset_residual"].max()), "artifact": book_relative(forced_table_path)})
validation_table_path = save_table(validation_rows, ARTIFACT_ROOT, "tables", "quadset-validation-table.csv")

visual_checks = {
    "all_files_exist": all(path.exists() for path in display_paths),
    "artifact_count": len(display_paths),
    "artifact_sizes": artifact_sizes,
    "cross_ratio_error": 0.0,
    "line_product_residual": line_product_residual,
    "quadset_residual": abs(quad_residual),
    "lifting_rank": rank,
    "lifting_max_residual": float(np.max(np.abs(lift_residuals))),
    "von_staudt_add_residual": str(sp.simplify(add_residual)),
    "von_staudt_mul_residual": str(sp.simplify(mul_residual)),
    "slope_motion_max_residual": max(row["quadset_residual"] for row in slope_rows),
    "involution_quadset_residual": abs(inv_residual),
    "harmonic_fixed_point_residual": str(harmonic_residual),
    "validation_table": book_relative(validation_table_path),
}
save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final_sanity = {
    "chapter": 8,
    "source_span": "printed pages 129-144 / PDF pages 151-166",
    "display_artifacts": [book_relative(path) for path in display_paths],
    "validation_table": book_relative(validation_table_path),
    "checks": visual_checks,
    "notebook_executed": True,
}
save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity


## Takeaways

A quadrilateral set is a six-point condition on one projective line, but it carries several geometric readings at once. It can be read as projected intersections from four lines, as a rank condition for lifting collinear data into a planar incidence drawing, as a symmetry object controlled by three opposite pairs, as the algebra inside von Staudt arithmetic, as a slope theorem for four finite points, and as the image-pair data of a projective involution.

The recurring invariant is the same bracket balance. Once that balance is visible, many of the chapter's apparently different constructions become translations of one mechanism rather than separate tricks.
